# Retrieval metrics and baselines: the real-library code of the lessons

Booster LEAP, Advanced RAG Engineering. The cells below are the library versions of the examples in the lessons (the lessons first build the same mechanism in plain Python in your browser). Each cell was run once and its output is saved below it.

<h1 style="color:#b45309">&#9888; First: File &rarr; Save a copy in Drive</h1>

<span style="font-size:20px">This notebook is read-only. Make your own copy now, then work in the copy (no GitHub login needed).</span>

**How to use it:** run the cells from top to bottom (Runtime, Run all, or Shift+Enter on each cell). Change one thing at a time and run again.

**What needs what:** the last cells download the small embedding model all-MiniLM-L6-v2 (about 90 MB; it needs internet but no GPU); scikit-learn and numpy are small. No secret or key is used in this notebook. The install line pins the versions that were used to produce the saved outputs; Colab may resolve the dependencies of those packages differently.

In [ ]:
%pip -q install scikit-learn==1.9.1 numpy==2.5.2 sentence-transformers==6.3.0

## Ground truth and labels

The same two kappa values with scikit-learn's cohen_kappa_score (real library, run in VS Code).

In [ ]:
# (query, passage, label of person A, label of person B); 1 = relevant, 0 = not relevant
pairs = [
    ("q09", "leave-annual", 1, 1),
    ("q09", "leave-carry", 1, 1),
    ("q09", "leave-sick", 1, 0),
    ("q09", "leave-parental", 0, 1),
    ("q10", "sec-data", 1, 1),
    ("q10", "faq-region", 1, 0),
    ("q10", "sec-passwords", 0, 0),
    ("q15", "kb-login-loop", 1, 1),
    ("q15", "kb-err4012", 1, 0),
    ("q15", "kb-err4013", 0, 0),
    ("q13", "kb-mail", 1, 1),
    ("q13", "kb-merge", 0, 0),
]
from sklearn.metrics import cohen_kappa_score

a = [x[2] for x in pairs]
b = [x[3] for x in pairs]
print("first set  kappa:", round(cohen_kappa_score(a, b), 3))
a2 = [0] * 17 + [1, 1, 0]
b2 = [0] * 17 + [0, 0, 1]
print("second set kappa:", round(cohen_kappa_score(a2, b2), 3))

first set  kappa: 0.333
second set kappa: -0.071


## Precision and recall at k

Precision, recall and F1 for q15 and q12 with scikit-learn's precision_recall_fscore_support (real library, run in VS Code).

In [ ]:
all_ids = [
    "leave-annual", "leave-sick", "leave-parental", "leave-carry",
    "leave-apply", "exp-meals", "exp-approval", "exp-travel",
    "sec-passwords", "sec-devices", "sec-data", "faq-seats",
    "faq-sso", "faq-trash", "faq-region", "faq-uptime",
    "run-sev", "run-rollback", "kb-err4012", "kb-err4013",
    "kb-err4021", "kb-err5031", "kb-err5032", "kb-err5103",
    "kb-login-loop", "kb-slow", "kb-mail", "kb-merge",
]
from sklearn.metrics import precision_recall_fscore_support

cases = {
    "q15": (['kb-err4012', 'kb-err4013', 'kb-login-loop'], ['kb-login-loop', 'kb-err4013', 'kb-err4021']),
    "q12": (['exp-approval', 'exp-travel'], ['exp-travel', 'faq-seats', 'kb-merge']),
}
for q, (relevant, top3) in cases.items():
    y_true = [1 if p in relevant else 0 for p in all_ids]     # one 0/1 label for each of the 28 passages
    y_pred = [1 if p in top3 else 0 for p in all_ids]         # the 3 returned passages are the predicted positives
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    by_hand = sum(x in relevant for x in top3) / 3
    print(f"{q}: precision {p:.3f} recall {r:.3f} f1 {f:.3f} (by hand precision@3 {by_hand:.3f})")

q15: precision 0.667 recall 0.667 f1 0.667 (by hand precision@3 0.667)
q12: precision 0.333 recall 0.500 f1 0.400 (by hand precision@3 0.333)


## Reciprocal rank and nDCG: does the order matter?

The lesson's nDCG against scikit-learn's ndcg_score for five saved rankings (run in VS Code).

In [ ]:
import math

import numpy as np
from sklearn.metrics import ndcg_score

ids = ['leave-annual', 'leave-sick', 'leave-parental', 'leave-carry', 'leave-apply', 'exp-meals', 'exp-approval', 'exp-travel', 'sec-passwords', 'sec-devices', 'sec-data', 'faq-seats', 'faq-sso', 'faq-trash', 'faq-region', 'faq-uptime', 'run-sev', 'run-rollback', 'kb-err4012', 'kb-err4013', 'kb-err4021', 'kb-err5031', 'kb-err5032', 'kb-err5103', 'kb-login-loop', 'kb-slow', 'kb-mail', 'kb-merge']
grades = {
    "q02": {"kb-err5031": 3, "kb-err5032": 1},
    "q08": {"leave-carry": 3, "leave-annual": 1},
    "q09": {"leave-annual": 3, "leave-carry": 2, "leave-sick": 1},
    "q10": {"sec-data": 3, "faq-region": 1},
}
runs = {
    ("keyword", "q09"): ["leave-annual", "kb-err4021", "leave-sick"],
    ("keyword", "q08"): ["leave-carry"],
    ("keyword", "q10"): [],
    ("embedding", "q09"): ["leave-annual", "leave-carry", "leave-sick", "leave-parental", "faq-uptime", "leave-apply"],
    ("embedding", "q02"): ["kb-err5032", "kb-err4021", "kb-err5031", "run-sev", "kb-err4013", "kb-err5103"],
}

def ndcg_lesson(ranked, grade, k):
    gains = [grade.get(p, 0) for p in ranked[:k]]
    ideal = sorted(grade.values(), reverse=True)[:k]
    dcg = lambda g: sum(x / math.log2(i + 1) for i, x in enumerate(g, 1))
    return dcg(gains) / dcg(ideal)

for (name, q), ranked in runs.items():
    y_true = np.array([[grades[q].get(p, 0) for p in ids]], dtype=float)    # one row (one query), one column per passage
    y_score = np.zeros((1, len(ids)))
    for place, p in enumerate(ranked):                                       # a higher score = a better place; passages not returned keep 0
        y_score[0, ids.index(p)] = len(ranked) - place
    mine, theirs = ndcg_lesson(ranked, grades[q], 3), ndcg_score(y_true, y_score, k=3)
    print(q, f"{name:9} returned {len(ranked)} passages  lesson {mine:.4f}  scikit-learn {theirs:.4f}")

q09 keyword   returned 3 passages  lesson 0.7350  scikit-learn 0.7350
q08 keyword   returned 1 passages  lesson 0.8262  scikit-learn 0.8378
q10 keyword   returned 0 passages  lesson 0.0000  scikit-learn 0.0838
q09 embedding returned 6 passages  lesson 1.0000  scikit-learn 1.0000
q02 embedding returned 6 passages  lesson 0.6885  scikit-learn 0.6885


## Synthetic test sets and paired comparison

The same tests with scipy: permutation_test with permutation_type="samples" flips the pairs and, as n_resamples is at least 2^16, enumerates all of them. The p-values equal the lesson's code. Check the version you use.

In [ ]:
first_rank = {
    "keyword": [1, 1, 1, 1, 1, 1, 1, 1, 1, None, None, None, None, None, 1, None],
    "embedding": [4, 1, 3, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    "mix": [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 7, 1, 1, 1],
}
import numpy as np
import scipy
from scipy.stats import permutation_test

def reciprocal(ranks):
    return np.array([0.0 if r is None else 1 / r for r in ranks])

print("scipy", scipy.__version__)
for x, y in [("mix", "embedding"), ("keyword", "embedding"), ("keyword", "mix")]:
    result = permutation_test((reciprocal(first_rank[x]), reciprocal(first_rank[y])), lambda a, b, axis: np.mean(a - b, axis=axis),
                              permutation_type="samples", n_resamples=2 ** 16, vectorized=True, alternative="two-sided")
    print(f"{x} - {y}: mean {result.statistic:+.4f}, p {result.pvalue:.4f}")

scipy 1.18.1
mix - embedding: mean +0.0350, p 0.8125
keyword - embedding: mean -0.2552, p 0.1250
keyword - mix: mean -0.2902, p 0.0312


## Notebook: measure retrieval from end to end

The plain functions and scikit-learn side by side for three metrics. Check the version you use.

In [ ]:
# query: (grades, keyword list, embedding list, mix list); each list is cut after 7 passages, enough for every first relevant rank
runs = {
    "q01": ({"kb-err5103": 3}, ["kb-err5103", "kb-err4012", "kb-err4021", "kb-err5031", "kb-err5032", "kb-err4013"], ["kb-err4013", "kb-err4021", "kb-err5032", "kb-err5103", "kb-err5031", "run-sev", "kb-err4012"], ["kb-err5103", "kb-err4021", "kb-err4013", "kb-err5032", "kb-err4012", "kb-err5031", "run-sev"]),
    "q02": ({"kb-err5031": 3, "kb-err5032": 1}, ["kb-err5031", "kb-err4012", "kb-err5103", "kb-err4021", "kb-err5032", "kb-err4013"], ["kb-err5032", "kb-err4021", "kb-err5031", "run-sev", "kb-err4013", "kb-err5103", "kb-err4012"], ["kb-err5031", "kb-err5032", "kb-err4021", "kb-err4012", "kb-err5103", "kb-err4013", "run-sev"]),
    "q03": ({"kb-err4012": 3}, ["kb-err4012", "kb-err5103", "kb-err4021", "kb-err5031", "kb-err5032", "kb-err4013"], ["kb-err4013", "kb-err4021", "kb-err4012", "kb-err5031", "kb-err5032", "sec-passwords", "kb-login-loop"], ["kb-err4012", "kb-err4021", "kb-err4013", "kb-err5031", "kb-err5032", "kb-err5103", "sec-passwords"]),
    "q04": ({"run-sev": 3}, ["run-sev", "kb-err4021", "leave-annual", "kb-err5031", "sec-devices", "exp-meals"], ["run-sev", "leave-apply", "kb-slow", "exp-travel", "kb-err5032", "kb-err4021", "kb-err5103"], ["run-sev", "kb-err4021", "kb-err5031", "sec-devices", "leave-annual", "exp-meals", "leave-apply"]),
    "q05": ({"faq-region": 3}, ["faq-region"], ["faq-region", "run-sev", "faq-seats", "kb-err5031", "faq-trash", "kb-merge", "exp-meals"], ["faq-region", "run-sev", "faq-seats", "kb-err5031", "faq-trash", "kb-merge", "exp-meals"]),
    "q06": ({"kb-slow": 3}, ["kb-slow", "faq-trash", "kb-merge", "faq-uptime", "leave-sick", "kb-err5031"], ["kb-merge", "kb-slow", "faq-trash", "kb-err5031", "faq-seats", "exp-approval", "exp-meals"], ["kb-slow", "kb-merge", "faq-trash", "kb-err5031", "faq-uptime", "leave-sick", "faq-seats"]),
    "q07": ({"exp-meals": 3}, ["exp-meals", "kb-err5031", "run-sev", "sec-devices"], ["exp-meals", "faq-seats", "leave-parental", "faq-uptime", "exp-travel", "leave-annual", "kb-merge"], ["exp-meals", "kb-err5031", "run-sev", "sec-devices", "faq-seats", "leave-parental", "faq-uptime"]),
    "q08": ({"leave-carry": 3, "leave-annual": 1}, ["leave-carry"], ["leave-carry", "run-sev", "leave-annual", "faq-trash", "kb-err4021", "leave-apply", "kb-login-loop"], ["leave-carry", "run-sev", "leave-annual", "faq-trash", "kb-err4021", "leave-apply", "kb-login-loop"]),
    "q09": ({"leave-annual": 3, "leave-carry": 2, "leave-sick": 1}, ["leave-annual", "kb-err4021", "leave-sick"], ["leave-annual", "leave-carry", "leave-sick", "leave-parental", "faq-uptime", "leave-apply", "run-sev"], ["leave-annual", "leave-sick", "kb-err4021", "leave-carry", "leave-parental", "faq-uptime", "leave-apply"]),
    "q10": ({"sec-data": 3, "faq-region": 1}, [], ["sec-data", "sec-passwords", "faq-region", "kb-merge", "faq-sso", "kb-err4021", "exp-approval"], ["sec-data", "sec-passwords", "faq-region", "kb-merge", "faq-sso", "kb-err4021", "exp-approval"]),
    "q11": ({"leave-parental": 3, "leave-apply": 1}, ["faq-trash"], ["leave-parental", "leave-apply", "leave-carry", "leave-annual", "leave-sick", "run-sev", "exp-meals"], ["faq-trash", "leave-parental", "leave-apply", "leave-carry", "leave-annual", "leave-sick", "run-sev"]),
    "q12": ({"exp-travel": 3, "exp-approval": 1}, [], ["exp-travel", "faq-seats", "kb-merge", "kb-err5031", "faq-trash", "kb-slow", "kb-login-loop"], ["exp-travel", "faq-seats", "kb-merge", "kb-err5031", "faq-trash", "kb-slow", "kb-login-loop"]),
    "q13": ({"kb-mail": 3, "kb-merge": 1}, ["faq-region", "run-sev", "leave-carry", "faq-seats", "kb-slow", "leave-sick"], ["kb-mail", "sec-data", "run-sev", "faq-region", "kb-slow", "kb-login-loop", "kb-err4021"], ["faq-region", "run-sev", "kb-slow", "leave-carry", "faq-seats", "leave-sick", "kb-mail"]),
    "q14": ({"kb-slow": 3}, [], ["kb-slow", "kb-login-loop", "kb-err4021", "kb-err5031", "run-sev", "kb-err4013", "run-rollback"], ["kb-slow", "kb-login-loop", "kb-err4021", "kb-err5031", "run-sev", "kb-err4013", "run-rollback"]),
    "q15": ({"kb-login-loop": 3, "kb-err4012": 1, "kb-err4013": 1}, ["kb-login-loop", "kb-merge", "kb-err5103"], ["kb-login-loop", "kb-err4013", "kb-err4021", "kb-err4012", "sec-passwords", "run-rollback", "sec-devices"], ["kb-login-loop", "kb-merge", "kb-err5103", "kb-err4013", "kb-err4021", "kb-err4012", "sec-passwords"]),
    "q16": ({"sec-devices": 3}, [], ["sec-devices", "sec-data", "leave-apply", "leave-carry", "leave-sick", "leave-annual", "sec-passwords"], ["sec-devices", "sec-data", "leave-apply", "leave-carry", "leave-sick", "leave-annual", "sec-passwords"]),
}
ids = ["leave-annual", "leave-sick", "leave-parental", "leave-carry", "leave-apply", "exp-meals", "exp-approval",
       "exp-travel", "sec-passwords", "sec-devices", "sec-data", "faq-seats", "faq-sso", "faq-trash", "faq-region",
       "faq-uptime", "run-sev", "run-rollback", "kb-err4012", "kb-err4013", "kb-err4021", "kb-err5031", "kb-err5032",
       "kb-err5103", "kb-login-loop", "kb-slow", "kb-mail", "kb-merge"]
import math

def precision(ranked, grades, k): return sum(p in grades for p in ranked[:k]) / k
def recall(ranked, grades, k): return sum(p in grades for p in ranked[:k]) / len(grades)
def rr(ranked, grades): return next((1 / i for i, p in enumerate(ranked, 1) if p in grades), 0.0)
def dcg(gains): return sum(g / math.log2(i + 1) for i, g in enumerate(gains, 1))
def ndcg(ranked, grades, k):
    return dcg([grades.get(p, 0) for p in ranked[:k]]) / dcg(sorted(grades.values(), reverse=True)[:k])

SYSTEMS = ["keyword", "embedding", "mix"]
METRICS = ["p@3", "r@3", "rr", "ndcg@3"]

def scores(system):
    """one row of four numbers per query"""
    rows = {}
    for qid, (grades, *lists) in runs.items():
        ranked = lists[SYSTEMS.index(system)]
        rows[qid] = dict(zip(METRICS, [precision(ranked, grades, 3), recall(ranked, grades, 3), rr(ranked, grades), ndcg(ranked, grades, 3)]))
    return rows

def mean(rows, metric):
    return sum(r[metric] for r in rows.values()) / len(rows)

import numpy as np
import sklearn
from sklearn.metrics import ndcg_score, precision_score, recall_score

def sklearn_scores(system):
    out = {"p@3": [], "r@3": [], "ndcg@3": []}
    for qid, (grades, *lists) in runs.items():
        ranked = lists[SYSTEMS.index(system)]
        y_true = np.array([[grades.get(p, 0) for p in ids]])
        y_score = np.zeros((1, len(ids)))                       # passages that the system did not list get score 0
        for position, p in enumerate(ranked):
            y_score[0, ids.index(p)] = len(ranked) - position + 1
        relevant = (y_true[0] > 0).astype(int)
        in_top3 = np.array([int(p in ranked[:3]) for p in ids])
        out["ndcg@3"].append(ndcg_score(y_true, y_score, k=3))
        out["p@3"].append(precision_score(relevant, in_top3, zero_division=0))
        out["r@3"].append(recall_score(relevant, in_top3, zero_division=0))
    return {m: float(np.mean(v)) for m, v in out.items()}

print("scikit-learn", sklearn.__version__)
print("system     metric     plain code  scikit-learn")
for system in SYSTEMS:
    plain, library = scores(system), sklearn_scores(system)
    for metric in ("p@3", "r@3", "ndcg@3"):
        print(f"{system:10s} {metric:8s} {mean(plain, metric):11.4f} {library[metric]:13.4f}")

scikit-learn 1.9.1
system     metric     plain code  scikit-learn
keyword    p@3           0.2292        0.3125
keyword    r@3           0.5000        0.5000
keyword    ndcg@3        0.5696        0.5932
embedding  p@3           0.4583        0.4583
embedding  r@3           0.8542        0.8542
embedding  ndcg@3        0.8299        0.8299
mix        p@3           0.4167        0.4167
mix        r@3           0.8438        0.8438
mix        ndcg@3        0.8689        0.8689


The real model against the saved top 3 for the 16 questions.

In [ ]:
docs = {
    "leave-annual": "Full-time employees receive 24 days of paid annual leave per calendar year. Public holidays are not counted.",
    "leave-sick": "Employees may take up to 10 paid sick days per calendar year. From the third consecutive sick day a medical certificate is required.",
    "leave-parental": "The primary caregiver receives 26 weeks of paid parental leave and the second caregiver receives 6 weeks.",
    "leave-carry": "Up to 5 unused annual leave days may be carried over and must be used by 31 March.",
    "leave-apply": "Request leave in the HR portal at least 5 working days before the first day of leave.",
    "exp-meals": "Daily meal limits in rupees: Tier 1 city 400 breakfast, 700 lunch, 1200 dinner. Tier 2 city 300, 500, 900. Tier 3 city 250, 400, 700.",
    "exp-approval": "Claims above 5000 rupees need a receipt and manager approval. Claims above 25000 rupees also need finance approval.",
    "exp-travel": "Book economy class for flights shorter than 6 hours. Business class needs written approval from the department head.",
    "sec-passwords": "Passwords must have at least 14 characters and must not be reused. Multi-factor authentication is required for email and the cloud console.",
    "sec-devices": "Report a lost or stolen laptop to the security team within 2 hours. Company laptops lock after 5 minutes without activity.",
    "sec-data": "Never copy customer data into chat tools, personal notes or public AI services.",
    "faq-seats": "The Starter plan includes up to 5 seats. Teams that need more seats move to the Team plan.",
    "faq-sso": "Single sign-on with SAML is available on the Business plan and above.",
    "faq-trash": "Deleted tickets stay in the trash for 30 days and are then removed for good.",
    "faq-region": "Customer data is stored in the Mumbai region by default. Business plan customers may choose Frankfurt.",
    "faq-uptime": "The Business plan commits to 99.9 percent monthly uptime, with a credit of 10 percent of the month's fee if it falls below.",
    "run-sev": "Sev 1: the service is down for most customers. Respond within 5 minutes. Sev 2: respond within 30 minutes.",
    "run-rollback": "If the problem started right after a release, roll back first. Command: orbit deploy rollback --service desk-api --to previous.",
    "kb-err4012": "Error ERR-4012 at sign-in: the SAML certificate has expired. An admin uploads the new certificate in Settings, then Security.",
    "kb-err4013": "Error ERR-4013 at sign-in: the password was typed wrongly too many times. The account unlocks by itself after 15 minutes.",
    "kb-err4021": "Error ERR-4021 at sign-in: the authenticator code is out of date. Check that the phone clock is set to automatic time.",
    "kb-err5031": "Error ERR-5031 on export: the export file is larger than 2 GB. Filter the tickets by date and export in several parts.",
    "kb-err5032": "Error ERR-5032 on export: the export link has expired after 24 hours. Start a new export in Settings, then Data.",
    "kb-err5103": "Error ERR-5103 on import: the CSV file has a column that is not in the template. Remove the extra column and upload again.",
    "kb-login-loop": "If the sign-in page keeps reloading, clear the browser cookies for orbitdesk.example and try again.",
    "kb-slow": "Pages load slowly when the inbox holds more than 10,000 open tickets. Archive old tickets to speed it up.",
    "kb-mail": "Customer emails missing from the inbox are usually in the spam filter. Add the sender to the allow list in Settings, then Mail.",
    "kb-merge": "Two tickets from the same customer can be merged. The older ticket keeps its number.",
}
# query: (text, the top 3 of the saved embedding ranking)
saved = {
    "q01": ("ERR-5103", ["kb-err4013", "kb-err4021", "kb-err5032"]),
    "q02": ("ERR-5031", ["kb-err5032", "kb-err4021", "kb-err5031"]),
    "q03": ("ERR-4012", ["kb-err4013", "kb-err4021", "kb-err4012"]),
    "q04": ("Sev 2 response time", ["run-sev", "leave-apply", "kb-slow"]),
    "q05": ("Frankfurt", ["faq-region", "run-sev", "faq-seats"]),
    "q06": ("10,000 tickets", ["kb-merge", "kb-slow", "faq-trash"]),
    "q07": ("Tier 2 dinner limit", ["exp-meals", "faq-seats", "leave-parental"]),
    "q08": ("31 March", ["leave-carry", "run-sev", "leave-annual"]),
    "q09": ("how much time off do I get each year", ["leave-annual", "leave-carry", "leave-sick"]),
    "q10": ("can I paste client information into ChatGPT", ["sec-data", "sec-passwords", "faq-region"]),
    "q11": ("I am expecting a baby, how long can I stay home", ["leave-parental", "leave-apply", "leave-carry"]),
    "q12": ("can I fly in a fancier seat", ["exp-travel", "faq-seats", "kb-merge"]),
    "q13": ("messages from customers do not show up", ["kb-mail", "sec-data", "run-sev"]),
    "q14": ("the app is very slow", ["kb-slow", "kb-login-loop", "kb-err4021"]),
    "q15": ("the website keeps showing the login again and again", ["kb-login-loop", "kb-err4013", "kb-err4021"]),
    "q16": ("what if my work computer is taken", ["sec-devices", "sec-data", "leave-apply"]),
}

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
ids = list(docs)
vectors = model.encode([docs[i] for i in ids], normalize_embeddings=True)       # length 1, so a dot product is the cosine
same = 0
for qid, (text, expected) in saved.items():
    cosines = vectors @ model.encode([text], normalize_embeddings=True)[0]
    order = sorted(range(len(ids)), key=lambda j: (-float(cosines[j]), ids[j]))
    top3 = [ids[j] for j in order[:3]]
    same += top3 == expected
    if top3 != expected:
        print(qid, "differs:", top3, "against", expected)
    if qid == "q09":
        print(qid, [(ids[j], round(float(cosines[j]), 3)) for j in order[:3]])
print("top 3 equal to the saved embedding ranking:", same, "of", len(saved), "queries")

q09 [('leave-annual', 0.48), ('leave-carry', 0.458), ('leave-sick', 0.363)]
top 3 equal to the saved embedding ranking: 16 of 16 queries
